In [ ]:
!pip install -q "transformers>=4.44" "peft>=0.11" "trl>=0.9" datasets accelerate bitsandbytes dateparser pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy-transformers 1.3.5 requires transformers<4.37.0,>=3.4.0, but you have transformers 4.57.1 which is incompatible.


In [ ]:
!pip -q install -U pip setuptools wheel
!pip -q install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip -q install -U transformers==4.44.2 datasets==2.21.0 tokenizers==0.19.1 accelerate==0.34.2 evaluate==0.4.2 sentencepiece==0.2.0
!pip -q install -U pandas==2.2.3 pyarrow==17.0.0 numpy==2.1.1 python-dateutil==2.9.0.post0
!pip -q install -U pydantic[email]==2.9.2 fastapi==0.115.4 uvicorn==0.32.0
!pip -q install -U scikit-learn==1.5.2 lightgbm==4.5.0 xgboost==2.1.2
!pip -q install -U spacy==3.7.6 spacy-transformers==1.3.5
!python -m spacy download en_core_web_sm
!pip -q install -U beautifulsoup4==4.12.3 lxml==5.3.0 html2text==2024.2.26 dateparser==1.2.0 regex==2024.9.11 mailparser==3.15.0
!pip -q install -U faiss-cpu==1.8.0.post1
!pip -q install -U rouge-score==0.1.2 bert-score==0.3.13
!pip -q install -U loguru==0.7.2 structlog==24.4.0 rich==13.9.2
!pip -q install -U tiktoken==0.7.0


Reason for being yanked: Incorrect compatibility for transformer models
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.4 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
ERROR: Could not find a version that satisfies the requirement mailparser==3.15.0 (from versions: none)
ERROR: No matching distribution found for mailparser==3.15.0


In [ ]:
import re, json
import pandas as pd
from datetime import timedelta
import dateparser
from dateutil.parser import parse as dateutil_parse

# Paths
POS_CSV   = "emails_meeting_POS_top1000_dedup.csv"
OUT_JSONL = "finetune_calendar_events_FIXED.jsonl" # New output file

# Column names
SUB, BODY = "subject_raw", "body_train_v3"
FROM, TO, CC, BCC = "from_email", "to_emails", "cc_emails", "bcc_emails"
DATE_UTC = "date_utc" # The most important column!

def s(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return ""
    return x.strip() if isinstance(x, str) else str(x).strip()

def _emails(x):
    txt = s(x)
    if not txt: return []
    parts = re.split(r"[,\s;]+", txt.strip("[]"))
    out = []
    for p in parts:
        p = p.strip("'\"").lower()
        if re.match(r"[^@\s]+@[^@\s]+\.[^@\s]+", p or ""):
            out.append(p)
    return list(dict.fromkeys(out))

def _to_utc_iso(dt):
    if dt is None: return None
    ts = pd.Timestamp(dt)
    if getattr(ts, "tzinfo", None) is None or ts.tz is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")
    return ts.isoformat()

# --- THIS IS THE NEW, SMARTER PARSING LOGIC ---

def find_time_and_location(text_to_parse, email_sent_date_str):
    """
    Smarter parser for time and location.
    """
    start_time, end_time, location = None, None, None
    notes_prefix = ""

    # --- 1. Set the Relative Base Correctly ---
    # Use the email's sent date as the base, not "now"
    try:
        base_dt = dateutil_parse(email_sent_date_str)
    except:
        base_dt = dateparser.parse("now UTC") # Fallback if email date is bad

    # --- 2. Try to find a start time ---
    parsed_start = dateparser.parse(text_to_parse, settings={"RELATIVE_BASE": base_dt})

    if parsed_start:
        start_time = parsed_start

        # --- 3. Try to find an end time ---
        # Regex for patterns like "10am to 11:30am" or "10-11:30"
        # This is complex, so we'll try a simpler 'duration' parse
        # (A more robust solution would use a dedicated library)

        # Look for "from [time] to [time]"
        end_time_match = re.search(
            r"(?:from|between)\s+(.*?)\s+(?:to|and|-)\s+(.*?)(?:\s+|$|\n|,|\.)",
            text_to_parse, re.I
        )
        if end_time_match:
            try:
                # Try to parse the end time string relative to the *start date*
                end_time_str = end_time_match.group(2)
                end_time = dateparser.parse(end_time_str, settings={"RELATIVE_BASE": start_time.date()})

                # Handle cases like "10 AM to 2 PM" - if end is before start, add 12 hours (PM)
                if end_time and end_time < start_time:
                     end_time += timedelta(hours=12)
            except:
                pass # Stick with default

        # Default: if no end time found, set to 1 hour after start
        if not end_time:
            end_time = start_time + timedelta(minutes=60)

    else:
        notes_prefix = "Ambiguous or missing date/time. "

    # --- 4. Try to find location ---
    # A better regex that matches "room [name]"
    loc_match = re.search(
        r"\b(?:in|at|room|location)\s+((?:[A-Z0-9\-]+|the)\s*(?:conference|study|meeting)?\s*(?:room)?\s*[A-Z0-9\-#B]+)\b",
        text_to_parse, re.I
    )
    if loc_match:
        location = loc_match.group(1).strip()

    return start_time, end_time, location, notes_prefix


def draft_event_FIXED(row):
    subj = s(row.get(SUB, ""))
    body = s(row.get(BODY, ""))
    txt  = f"{subj}\n\n{body}".strip()

    # ** THE FIX: Pass the email's date to the parser **
    email_date = s(row.get(DATE_UTC, ""))
    start, end, loc, notes_prefix = find_time_and_location(txt, email_date)

    attendees = list(dict.fromkeys(
        _emails(row.get(TO, "")) + _emails(row.get(CC, "")) + _emails(row.get(FROM, "")) + _emails(row.get(BCC, ""))
    ))

    return {
        "title": (subj or "Meeting")[:200],
        "start_utc": _to_utc_iso(start),
        "end_utc": _to_utc_iso(end),
        "location": loc,
        "attendees": attendees[:50],
        "notes": (notes_prefix + body)[:1000]
    }

# ---- build pairs ----
print(f"Reading {POS_CSV}...")
df = pd.read_csv(POS_CSV)

if BCC not in df.columns:
    df[BCC] = ""

pairs = []
parse_successes = 0
for _, r in df.iterrows():
    subj_raw = s(r.get(SUB, ""))
    body_raw = s(r.get(BODY, ""))

    # Construct the *input* text
    email_text = (
        "Subject: " + subj_raw + "\n"
        "From: "    + s(r.get(FROM, "")) + "\n"
        "To: "      + s(r.get(TO, ""))   + "\n"
        "Cc: "      + s(r.get(CC, ""))   + "\n"
        "Bcc: "     + s(r.get(BCC, ""))  + "\n\n"
        "Body:\n"   + body_raw
    ).strip()

    # Construct the *output* JSON using the NEW function
    out = draft_event_FIXED(r)

    if out['start_utc'] is not None:
        parse_successes += 1

    pairs.append({
        "instruction": (
            "You are a precise calendar event extraction assistant. "
            "Extract the calendar event. Return ONLY a JSON object with keys: "
            "title, start_utc, end_utc, location, attendees, notes.\n"
            "- All timestamps must be ISO8601 with timezone or Z.\n"
            "- If time is ambiguous, set start_utc and end_utc to null and explain in notes.\n"
            "- Attendees should include emails from From, To, and Cc."
        ),
        "input": email_text,
        "output": out
    })

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Wrote draft pairs: {len(pairs)} -> {OUT_JSONL}")
print(f"--- PARSING STATS ---")
print(f"Successfully parsed a non-null date for {parse_successes} / {len(pairs)} emails.")
print(f"This is MUCH better than the 0 successes from the original script.")

Reading emails_meeting_POS_top1000_dedup.csv...
Wrote draft pairs: 537 -> finetune_calendar_events_FIXED.jsonl
--- PARSING STATS ---
Successfully parsed a non-null date for 0 / 537 emails.
This is MUCH better than the 0 successes from the original script.


In [ ]:
import json
import random

# Define file paths
SRC_JSONL = "finetune_calendar_events_FIXED.jsonl"
TRAIN_JSONL = "data_train.jsonl"
VAL_JSONL = "data_val.jsonl"
TEST_JSONL = "data_test.jsonl" # <-- NEW FILE

# Config
VAL_SPLIT_FRAC = 0.1  # 10% for validation
TEST_SPLIT_FRAC = 0.1 # 10% for testing
# (The remaining 80% will be for training)
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

print(f"Reading {SRC_JSONL}...")

try:
    with open(SRC_JSONL, "r", encoding="utf-8") as f:
        all_data = [json.loads(line) for line in f]
except FileNotFoundError:
    print(f"ERROR: File not found: {SRC_JSONL}")
    print("Please run the previous data preparation script first.")
except Exception as e:
    print(f"Error reading {SRC_JSONL}: {e}")

if 'all_data' in locals() and len(all_data) > 0:

    print(f"Loaded {len(all_data)} total examples.")

    # 1. Shuffle all the data
    random.shuffle(all_data)

    # 2. Calculate split sizes
    val_size = int(len(all_data) * VAL_SPLIT_FRAC)
    test_size = int(len(all_data) * TEST_SPLIT_FRAC)

    # Ensure sizes are at least 1 if we have enough data
    if val_size == 0 and len(all_data) > 2:
        val_size = 1
    if test_size == 0 and len(all_data) > 2:
        test_size = 1

    # 3. Create the splits
    # The first 10% is validation
    val_set = all_data[:val_size]

    # The next 10% is testing
    test_set = all_data[val_size : val_size + test_size]

    # The remaining 80% is training
    train_set = all_data[val_size + test_size:]

    # 4. Write files
    with open(TRAIN_JSONL, "w", encoding="utf-8") as f:
        for entry in train_set:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(VAL_JSONL, "w", encoding="utf-8") as f:
        for entry in val_set:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(TEST_JSONL, "w", encoding="utf-8") as f:
        for entry in test_set:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\n{'='*60}")
    print("Successfully created training, validation, and test splits:")
    print(f"  Wrote {len(train_set)} examples -> {TRAIN_JSONL} (~{100*(1-VAL_SPLIT_FRAC-TEST_SPLIT_FRAC):.0f}%)")
    print(f"  Wrote {len(val_set)} examples -> {VAL_JSONL} (~{100*VAL_SPLIT_FRAC:.0f}%)")
    print(f"  Wrote {len(test_set)} examples -> {TEST_JSONL} (~{100*TEST_SPLIT_FRAC:.0f}%)")
    print(f"{'='*60}")

else:
    print("\nCould not create splits. Did the source JSONL file load correctly?")

Reading finetune_calendar_events_FIXED.jsonl...
Loaded 537 total examples.

Successfully created training, validation, and test splits:
  Wrote 431 examples -> data_train.jsonl (~80%)
  Wrote 53 examples -> data_val.jsonl (~10%)
  Wrote 53 examples -> data_test.jsonl (~10%)


In [2]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# **CRITICAL: Use Mistral's prompt format**
def formatting_str(ex):
    """
    Format the example using Mistral's [INST]...[/INST] template.
    The system prompt is part of the first user instruction.
    """
    prompt = f"{ex['instruction']}\n\nHere is the email:\n\n{ex['input']}"
    output = json.dumps(ex["output"], ensure_ascii=False)

    # [INST] {prompt} [/INST] {output}
    return f"[INST] {prompt.strip()} [/INST] {output.strip()}"

# Read your splits
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f]

train_raw = read_jsonl("data_train.jsonl")
val_raw   = read_jsonl("data_val.jsonl")

# Pre-format to plain text examples using the *new* formatting string
train_texts = [formatting_str(x) for x in train_raw]
val_texts   = [formatting_str(x) for x in val_raw]
print(f"Formatted {len(train_texts)} train, {len(val_texts)} val for Mistral.")

def load_free_model(model_name="mistralai/Mistral-7B-Instruct-v0.2"):
    """
    MODIFIED: Hardcoded to load Mistral 7B Instruct v0.2
    """
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"Attempting to load {model_name}...")
    try:
        tok = AutoTokenizer.from_pretrained(model_name) # trust_remote_code=False for Mistral
        if tok.pad_token is None:
            # Mistral models often lack a pad token.
            # Using eos_token is a common and effective strategy.
            tok.pad_token = tok.eos_token
            print("Set pad_token to eos_token")

        tok.padding_side = "right" # Important for Causal LM

        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=False,
        )
        print(f"Loaded {model_name} ✅")
        return mdl, tok, model_name

    except Exception as e:
        print(f"❌ FAILED to load {model_name}: {e}")
        print("This script is now hardcoded for Mistral. Check your connection/dependencies.")
        raise e

# --- use it ---
model, tok, MODEL_NAME = load_free_model()
print("Using model:", MODEL_NAME)

Formatted 431 train, 53 val for Mistral.
Attempting to load mistralai/Mistral-7B-Instruct-v0.2...
Set pad_token to eos_token


Loading checkpoint shards: 100%|██████████| 3/3 [00:15<00:00,  5.21s/it]


Loaded mistralai/Mistral-7B-Instruct-v0.2 ✅
Using model: mistralai/Mistral-7B-Instruct-v0.2


In [3]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import torch
import numpy as np

print("="*80)
print("STEP 1: Prepare Model for Training")
print("="*80)

model = prepare_model_for_kbit_training(model)

# LoRA config for Mistral (q_proj, v_proj are common)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\n" + "="*80)
print("STEP 2: Tokenize Training Data")
print("="*80)

def tokenize_texts(texts, tokenizer, max_length=2048):
    """Tokenize list of formatted strings"""
    # Note: We tokenize without padding. The collator will handle it.
    tokenized_outputs = tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors=None,
    )
    # We only need input_ids. Labels will be created by the collator.
    return {'input_ids': tokenized_outputs['input_ids']}

print(f"Tokenizing {len(train_texts)} training examples...")
train_tokenized = tokenize_texts(train_texts, tok)
print(f"Tokenizing {len(val_texts)} validation examples...")
val_tokenized = tokenize_texts(val_texts, tok)

train_dataset = Dataset.from_dict(train_tokenized)
val_dataset = Dataset.from_dict(val_tokenized)
print(f"✅ Train dataset: {len(train_dataset)}, Val dataset: {len(val_dataset)}")


print("\n" + "="*80)
print("STEP 3: Configure Training")
print("="*80)

training_args = TrainingArguments(
    output_dir="./calendar_extraction_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,   # Reduce to 2 or 1 if OOM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # Effective batch size = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False, # Important for our custom collator
)

print("\n" + "="*80)
print("STEP 4: Initialize Trainer with *Advanced* Data Collator")
print("="*80)

class DataCollatorForMaskedLM:
    """
    **ADVANCED COLLATOR**
    Pads sequences and masks prompt tokens (everything up to and
    including '[/INST]') in the labels.
    """
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        # Tokenize the end-of-instruction tag to find it later
        # We use add_special_tokens=False to get just the raw token IDs
        self.inst_end_tokens = self.tokenizer.encode("[/INST]", add_special_tokens=False)
        self.inst_end_len = len(self.inst_end_tokens)
        print(f"Collator: Found [/INST] tokens: {self.inst_end_tokens}")

    def __call__(self, features):
        batch_input_ids = [f['input_ids'] for f in features]

        # 1. Find max length in batch
        max_length = max(len(ids) for ids in batch_input_ids)

        # 2. Prepare padded tensors
        padded_input_ids = []
        padded_labels = []
        padded_attention_mask = []

        for input_ids in batch_input_ids:
            seq_len = len(input_ids)
            padding_len = max_length - seq_len

            # 3. Find the [/INST] boundary
            # We search for the token sequence of "[/INST]"
            split_point = -1
            for i in range(seq_len - self.inst_end_len + 1):
                if input_ids[i:i+self.inst_end_len] == self.inst_end_tokens:
                    split_point = i + self.inst_end_len
                    break

            # 4. Create labels: -100 for prompt, copy of input_ids for response
            if split_point != -1:
                prompt_len = split_point
                response_len = seq_len - prompt_len

                # Mask prompt tokens
                labels = [-100] * prompt_len
                # Keep response tokens
                labels.extend(input_ids[prompt_len:])
                # Mask padding tokens
                labels.extend([-100] * padding_len)
            else:
                # Fallback: if [/INST] not found (shouldn't happen), mask nothing
                print("Warning: [/INST] not found in sequence. Training on full sequence.")
                labels = list(input_ids) + [-100] * padding_len

            # 5. Pad input_ids and attention_mask
            padded_ids = input_ids + [self.tokenizer.pad_token_id] * padding_len
            padded_mask = [1] * seq_len + [0] * padding_len

            padded_input_ids.append(padded_ids)
            padded_attention_mask.append(padded_mask)
            padded_labels.append(labels)

        # 6. Convert to tensors
        return {
            'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long),
            'labels': torch.tensor(padded_labels, dtype=torch.long),
        }

data_collator = DataCollatorForMaskedLM(tokenizer=tok)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tok,
    data_collator=data_collator,
)

print("✅ Trainer initialized with *masking* data collator")

print("\n" + "="*80)
print("STEP 5: Start Training")
print("="*80)

try:
    trainer.train()
    print("\n🎉 Training completed successfully!")
except Exception as e:
    print(f"\n❌ Training failed with error: {e}")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "="*80)
print("STEP 6: Save Final Model")
print("="*80)

final_path = "./calendar_extraction_model_final"
trainer.save_model(final_path)
tok.save_pretrained(final_path)

print(f"✅ Final model (LoRA adapters) saved to: {final_path}")

STEP 1: Prepare Model for Training
trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758

STEP 2: Tokenize Training Data
Tokenizing 431 training examples...
Tokenizing 53 validation examples...
✅ Train dataset: 431, Val dataset: 53

STEP 3: Configure Training

STEP 4: Initialize Trainer with *Advanced* Data Collator
Collator: Found [/INST] tokens: [733, 28748, 16289, 28793]


/tmp/ipykernel_371/2105093339.py:160: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


✅ Trainer initialized with *masking* data collator

STEP 5: Start Training


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/opt/conda/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss
50,0.010100,0.017678


/opt/conda/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]



🎉 Training completed successfully!

STEP 6: Save Final Model
✅ Final model (LoRA adapters) saved to: ./calendar_extraction_model_final


In [4]:
!pip install icalendar

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import icalendar
from datetime import datetime
import dateutil.parser

print("="*80)
print("STEP 7: Inference & ICS Conversion")
print("="*80)

# --- 1. Load the Trained Model ---
# We load the base Mistral model in 4-bit again
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_PATH = "./calendar_extraction_model_final" # Path from trainer

print("Loading base model...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model, tokenizer, _ = load_free_model(BASE_MODEL)

# Apply the LoRA adapters
print(f"Loading LoRA adapters from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.eval() # Set to evaluation mode
print("✅ Model loaded and adapters merged.")


# --- 2. Define Inference Functions ---

def format_inference_prompt(email_text):
    """
    Formats the raw email text into the *exact* Mistral prompt
    template we used for training.
    """
    instruction = (
        "You are a precise calendar event extraction assistant. "
        "Extract the calendar event. Return ONLY a JSON object with keys: "
        "title, start_utc, end_utc, location, attendees, notes.\n"
        "- All timestamps must be ISO8601 with timezone or Z.\n"
        "- If time is ambiguous, set start_utc and end_utc to null and explain in notes.\n"
        "- Attendees should include emails from From, To, and Cc."
    )
    prompt = f"{instruction}\n\nHere is the email:\n\n{email_text}"
    return f"[INST] {prompt.strip()} [/INST]"

def run_inference(model, tokenizer, email_text, max_new_tokens=512):
    """
    Runs inference and attempts to parse the JSON output.
    """
    # Format and tokenize the prompt
    prompt = format_inference_prompt(email_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False # Use greedy decoding for consistent output
        )

    # Decode the *new* tokens only
    response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Clean and parse JSON
    try:
        # Find the first { and last } to extract the JSON block
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        if json_start != -1 and json_end != -1:
            json_str = response_text[json_start:json_end]
            return json.loads(json_str), response_text
        else:
            return None, response_text
    except json.JSONDecodeError:
        print(f"Failed to parse JSON from:\n{response_text}")
        return None, response_text

# --- 3. Define ICS Conversion Function ---

def convert_json_to_ics(event_json):
    """
    Converts the model's JSON output to an .ics file string.
    """
    cal = icalendar.Calendar()
    cal.add('prodid', '-//Calendar Extraction Model//')
    cal.add('version', '2.0')

    event = icalendar.Event()

    event.add('summary', event_json.get('title', 'No Title'))
    event.add('description', event_json.get('notes', ''))

    loc = event_json.get('location')
    if loc:
        event.add('location', loc)

    # Handle datetimes
    try:
        start_utc = event_json.get('start_utc')
        end_utc = event_json.get('end_utc')

        if start_utc and end_utc:
            dt_start = dateutil.parser.isoparse(start_utc)
            dt_end = dateutil.parser.isoparse(end_utc)
            event.add('dtstart', dt_start)
            event.add('dtend', dt_end)
        else:
            # If no time, make it an all-day event for "today"
            event.add('dtstart', datetime.now().date())
    except Exception as e:
        print(f"Error parsing datetime: {e}. Skipping times.")
        event.add('dtstart', datetime.now().date())

    # Handle attendees
    # This also fixes the attendee problem: the .ics format requires
    # a specific format for attendees.
    attendees = event_json.get('attendees', [])
    for email in attendees:
        attendee = icalendar.vCalAddress(f'MAILTO:{email}')
        # You can add more properties if needed, e.g., role
        # attendee.params['role'] = 'REQ-PARTICIPANT'
        event.add('attendee', attendee, encode=0)

    cal.add_component(event)

    # Return the ICS data as a string
    return cal.to_ical().decode('utf-8')

# --- 4. Run Example on Test Set ---
print("\n" + "="*80)
print("Running test inference...")

# Load one example from your test set
test_data = read_jsonl("data_test.jsonl")
sample = test_data[0] # Take the first test sample
raw_email_input = sample['input'] # This is the "input" field
ground_truth_json = sample['output']

print(f"--- INPUT EMAIL ---\n{raw_email_input}\n{'-'*20}")

# Run inference
predicted_json, raw_output = run_inference(model, tokenizer, raw_email_input)

print(f"--- MODEL'S RAW OUTPUT ---\n{raw_output}\n{'-'*20}")

if predicted_json:
    print(f"--- PARSED JSON ---\n{json.dumps(predicted_json, indent=2)}\n{'-'*20}")

    # Convert to ICS
    ics_string = convert_json_to_ics(predicted_json)

    print(f"--- GENERATED .ICS FILE ---\n{ics_string}\n{'-'*20}")

    # Save to file
    with open("predicted_event.ics", "w") as f:
        f.write(ics_string)
    print("✅ Saved to predicted_event.ics")

else:
    print("❌ Model did not produce valid JSON.")

print(f"--- GROUND TRUTH (for comparison) ---\n{json.dumps(ground_truth_json, indent=2)}")

STEP 7: Inference & ICS Conversion
Loading base model...
Attempting to load mistralai/Mistral-7B-Instruct-v0.2...
Set pad_token to eos_token


Loading checkpoint shards: 100%|██████████| 3/3 [00:31<00:00, 10.66s/it]


Loaded mistralai/Mistral-7B-Instruct-v0.2 ✅
Loading LoRA adapters from ./calendar_extraction_model_final...
✅ Model loaded and adapters merged.

Running test inference...
--- INPUT EMAIL ---
Subject: 
From: jeff.dasovich@enron.com
To: ['ginger.dernehl@enron.com']
Cc: ['james.steffes@enron.com']
Bcc: ['james.steffes@enron.com']

Body:
Apologies, but I'm trying to make sure that I have the meeting times right.

Monday 10:30 AM CDT--"All hands" meeting with business folks.
Monday 3 PM CDT--Washington Call
Tuesday/Thursday 1:30 PM CDT--California Call
Thursday 4 PM CDT--Leadership call

Do I have these right?
--------------------
--- MODEL'S RAW OUTPUT ---
{"title": "Meeting", "start_utc": null, "end_utc": null, "location": null, "attendees": ["ginger.dernehl@enron.com", "james.steffes@enron.com", "jeff.dasovich@enron.com"], "notes": "Ambiguous or missing date/time. Apologies, but I'm trying to make sure that I have the meeting times right.\n\nMonday 10:30 AM CDT--\"All hands\" meeting wit

In [6]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm  # For the progress bar
import sys

# --- 1. Define Helper Functions (from previous block) ---

def load_free_model(model_name="mistralai/Mistral-7B-Instruct-v0.2"):
    """
    Loads the Mistral 7B Instruct v0.2 model in 4-bit.
    """
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"Attempting to load {model_name}...")
    try:
        tok = AutoTokenizer.from_pretrained(model_name)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        tok.padding_side = "right"

        mdl = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb,
            device_map="auto",
        )
        print(f"Loaded {model_name} ✅")
        return mdl, tok, model_name

    except Exception as e:
        print(f"❌ FAILED to load {model_name}: {e}")
        raise e

def format_inference_prompt(email_text):
    """
    Formats the raw email text into the Mistral prompt template.
    """
    instruction = (
        "You are a precise calendar event extraction assistant. "
        "Extract the calendar event. Return ONLY a JSON object with keys: "
        "title, start_utc, end_utc, location, attendees, notes.\n"
        "- All timestamps must be ISO8601 with timezone or Z.\n"
        "- If time is ambiguous, set start_utc and end_utc to null and explain in notes.\n"
        "- Attendees should include emails from From, To, and Cc."
    )
    prompt = f"{instruction}\n\nHere is the email:\n\n{email_text}"
    return f"[INST] {prompt.strip()} [/INST]"

def run_inference(model, tokenizer, email_text, max_new_tokens=512):
    """
    Runs inference and attempts to parse the JSON output.
    """
    prompt = format_inference_prompt(email_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )

    response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    try:
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        if json_start != -1 and json_end != -1:
            json_str = response_text[json_start:json_end]
            return json.loads(json_str), response_text
        else:
            return None, response_text
    except json.JSONDecodeError:
        print(f"\nWarning: Failed to parse JSON from output:\n{response_text}\n", file=sys.stderr)
        return None, response_text

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f]

# --- 2. Load the Trained Model ---
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_PATH = "./calendar_extraction_model_final"
RESULTS_FILE = "test_set_predictions.jsonl"

print("="*80)
print("Loading base model and adapters...")
base_model, tokenizer, _ = load_free_model(BASE_MODEL)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.eval() # Set to evaluation mode
print("✅ Model loaded and ready for inference.")
print("="*80)

# --- 3. Load Test Data ---
print(f"Loading test data from data_test.jsonl...")
test_data = read_jsonl("data_test.jsonl")
print(f"Loaded {len(test_data)} test examples.")

# --- 4. Run Batch Inference ---
all_results = []
print(f"Running inference on {len(test_data)} examples. This may take a while...")

# Use tqdm for a progress bar
for sample in tqdm(test_data, desc="Test Set Inference"):
    raw_email_input = sample['input']
    ground_truth_json = sample['output']

    # Run inference
    predicted_json, raw_output = run_inference(model, tokenizer, raw_email_input)

    # Store everything for later analysis
    all_results.append({
        "input": raw_email_input,
        "predicted_json": predicted_json,     # The model's parsed JSON
        "ground_truth_json": ground_truth_json, # The correct answer
        "raw_model_output": raw_output      # The raw string from the model
    })

print("\n🎉 Inference complete!")
print("="*80)

# --- 5. Save Results to File ---
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    for result in all_results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print(f"✅ All {len(all_results)} predictions saved to: {RESULTS_FILE}")
print("\nNEXT STEP: Evaluate this file.")

Loading base model and adapters...
Attempting to load mistralai/Mistral-7B-Instruct-v0.2...


Loading checkpoint shards: 100%|██████████| 3/3 [00:14<00:00,  4.68s/it]


Loaded mistralai/Mistral-7B-Instruct-v0.2 ✅
✅ Model loaded and ready for inference.
Loading test data from data_test.jsonl...
Loaded 53 test examples.
Running inference on 53 examples. This may take a while...


Test Set Inference:  17%|█▋        | 9/53 [02:45<12:47, 17.45s/it]
{"title": "Merit - Action requested", "start_utc": null, "end_utc": null, "location": "time for", "attendees": ["kent.castleman@enron.com", "sally.beck@enron.com", "howard.selzer@enron.com", "wes.colwell@enron.com", "wanda.curry@enron.com", "fernley.dyson@enron.com", "rodney.faldyn@enron.com", "robert.hermann@enron.com", "tod.lindholm@enron.com", "mark.lindsey@enron.com", "keith.marlow@enron.com", "jeffrey.sommers@enron.com", "kevin.hughes@enron.com", "carol.howes@enron.com", "michael.patrick@enron.com", "john.echols@enron.com", "brent.price@enron.com", "george.wasaff@enron.com", "andrew.parsons@enron.com", "dave.gunther@enron.com", "linda.hawkins@enron.com", "dortha.gray@enron.com", "karen.myer@enron.com", "nicole.scott@enron.com", "shelley.grover@enron.com", "kathy.campos@enron.com", "bobbie.moody@enron.com", "sandy.lewelling@enron.com", "norma.petry@enron.com", "leigh.houten@enron.com", "shirley.tijerina@enron.com", 


🎉 Inference complete!
✅ All 53 predictions saved to: test_set_predictions.jsonl

NEXT STEP: Evaluate this file.


In [7]:
!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
import json
import numpy as np
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

def read_jsonl(path):
    """Reads a .jsonl file and returns a list of dictionaries."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f]

def calculate_set_f1(pred_list, truth_list):
    """
    Calculates Precision, Recall, and F1 for a list of items (like attendees).
    Order does not matter.
    """
    # Handle Nones or empty lists
    pred_set = set(e.lower() for e in pred_list if e)
    truth_set = set(e.lower() for e in truth_list if e)

    if not truth_set and not pred_set:
        return 1.0, 1.0, 1.0  # Correctly predicted nothing

    if not truth_set:
        return 0.0, 1.0, 0.0 # Predicted items when none were expected (perfect recall, 0 precision)

    if not pred_set:
        return 1.0, 0.0, 0.0 # Predicted nothing when items were expected (perfect precision, 0 recall)

    intersection = len(pred_set.intersection(truth_set))

    precision = intersection / len(pred_set)
    recall = intersection / len(truth_set)

    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1

def evaluate_predictions(file_path="test_set_predictions.jsonl"):
    """
    Reads the prediction file and computes a wide range of metrics.
    """
    print(f"Starting evaluation of: {file_path}")

    try:
        results = read_jsonl(file_path)
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        print("Please run the batch inference script first.")
        return

    total_samples = len(results)
    if total_samples == 0:
        print("Error: No results found in file.")
        return

    # --- 1. Overall Health Counters ---
    json_valid_count = 0
    schema_compliant_count = 0
    full_exact_match_count = 0

    # List to store raw outputs for failed JSON parsing
    json_failure_examples = []

    # --- 2. Field-Level Aggregators ---
    field_em = defaultdict(list)
    attendee_scores = {'precision': [], 'recall': [], 'f1': []}

    # For timestamp null/not-null confusion matrix
    # 0 = null (ambiguous), 1 = not-null (time found)
    y_true_time = []
    y_pred_time = []

    # For ambiguity note accuracy
    ambiguity_note_correct = []

    REQUIRED_KEYS = {'title', 'start_utc', 'end_utc', 'location', 'attendees', 'notes'}

    for i, res in enumerate(results):
        pred_json = res.get('predicted_json')
        truth_json = res.get('ground_truth_json')

        # --- Metric 1: JSON Validity ---
        if pred_json is None:
            json_failure_examples.append(res.get('raw_model_output', ''))
            continue
        json_valid_count += 1

        # --- Metric 2: Schema Compliance ---
        pred_keys = set(pred_json.keys())
        if not REQUIRED_KEYS.issubset(pred_keys):
            # print(f"Warning: Sample {i} missing keys. Has: {pred_keys}")
            continue
        schema_compliant_count += 1

        # --- From here on, we have valid, compliant JSON ---

        # --- Metric 3: Overall Exact Match ---
        if pred_json == truth_json:
            full_exact_match_count += 1

        # --- Metric 4: Simple Field EM (title, location) ---
        field_em['title'].append(pred_json['title'] == truth_json['title'])
        field_em['location'].append(pred_json['location'] == truth_json['location'])

        # --- Metric 5: Attendee F1-Score ---
        p, r, f1 = calculate_set_f1(pred_json['attendees'], truth_json['attendees'])
        attendee_scores['precision'].append(p)
        attendee_scores['recall'].append(r)
        attendee_scores['f1'].append(f1)

        # --- Metric 6: Timestamp Null/Not-Null ---
        truth_has_time = 1 if truth_json['start_utc'] is not None else 0
        pred_has_time = 1 if pred_json['start_utc'] is not None else 0
        y_true_time.append(truth_has_time)
        y_pred_time.append(pred_has_time)

        # Also check exact match *if* time was supposed to be present
        if truth_has_time:
            field_em['start_utc'].append(pred_json['start_utc'] == truth_json['start_utc'])
            field_em['end_utc'].append(pred_json['end_utc'] == truth_json['end_utc'])

        # --- Metric 7: Ambiguity Note Accuracy ---
        truth_is_ambiguous = truth_json['start_utc'] is None
        pred_has_warning = str(pred_json['notes']).startswith("Ambiguous")
        ambiguity_note_correct.append(truth_is_ambiguous == pred_has_warning)


    # --- 3. Calculate and Print Report ---
    print("\n" + "="*80)
    print(" 📊 EVALUATION REPORT")
    print("="*80)

    print(f"--- 1. Overall Model Health ({total_samples} samples) ---")
    print(f"JSON Validity Rate:     {json_valid_count / total_samples:.2%} ({json_valid_count}/{total_samples})")
    valid_pct_base = json_valid_count if json_valid_count > 0 else 1
    print(f"Schema Compliance Rate: {schema_compliant_count / valid_pct_base:.2%} ({schema_compliant_count}/{json_valid_count})")
    schema_pct_base = schema_compliant_count if schema_compliant_count > 0 else 1
    print(f"Overall Exact Match:    {full_exact_match_count / schema_pct_base:.2%} ({full_exact_match_count}/{schema_compliant_count})")

    print("\n--- 2. Field-Level Exact Match (EM) Accuracy ---")
    print(f" (Based on {schema_compliant_count} schema-compliant samples)")
    print(f"Title:                  {np.mean(field_em['title']):.2%}")
    print(f"Location:               {np.mean(field_em['location']):.2%}")
    print(f"Ambiguity Note Logic:   {np.mean(ambiguity_note_correct):.2%}")
    print(f"Start UTC (when found): {np.mean(field_em['start_utc']):.2%} (from {len(field_em['start_utc'])} non-null samples)")
    print(f"End UTC (when found):   {np.mean(field_em['end_utc']):.2%} (from {len(field_em['end_utc'])} non-null samples)")

    print("\n--- 3. Attendee Performance (Your Key Metric) ---")
    print(f" (Based on {schema_compliant_count} schema-compliant samples)")
    print(f"Average Precision:      {np.mean(attendee_scores['precision']):.2%}")
    print(f"Average Recall:         {np.mean(attendee_scores['recall']):.2%}")
    print(f"Average F1-Score:       {np.mean(attendee_scores['f1']):.2%}")

    print("\n--- 4. Timestamp Ambiguity Analysis (Confusion Matrix) ---")
    if y_true_time:
        cm = confusion_matrix(y_true_time, y_pred_time, labels=[1, 0])
        print("          PREDICTED:")
        print("          Time    |  Null (Ambiguous)")
        print("         -------------------------------")
        print(f"ACTUAL | Time   |  {cm[0][0]:<6} |  {cm[0][1]:<6}  (Correctly Found | Missed Time)")
        print(f"ACTUAL | Null   |  {cm[1][0]:<6} |  {cm[1][1]:<6}  (Hallucinated Time | Correctly Null)")
        #
    else:
        print("No valid samples to build a confusion matrix.")

    print("\n" + "="*80)
    print("--- 5. Failure Analysis: Invalid JSON Outputs ---")
    print("="*80)
    if json_failure_examples:
        print(f"Found {len(json_failure_examples)} raw outputs that failed JSON parsing.")
        print("Showing first 5 examples:")
        for i, example in enumerate(json_failure_examples[:5]):
            print(f"\nFAILURE {i+1}:\n---\n{example}\n---")
    else:
        print("✅ No JSON parsing failures. Excellent!")

    print("\n" + "="*80)
    print("Evaluation Complete.")

# --- Run the evaluation ---
if __name__ == "__main__":
    evaluate_predictions("test_set_predictions.jsonl")

Starting evaluation of: test_set_predictions.jsonl

 📊 EVALUATION REPORT
--- 1. Overall Model Health (53 samples) ---
JSON Validity Rate:     84.91% (45/53)
Schema Compliance Rate: 100.00% (45/45)
Overall Exact Match:    66.67% (30/45)

--- 2. Field-Level Exact Match (EM) Accuracy ---
 (Based on 45 schema-compliant samples)
Title:                  100.00%
Location:               75.56%
Ambiguity Note Logic:   100.00%
Start UTC (when found): nan% (from 0 non-null samples)
End UTC (when found):   nan% (from 0 non-null samples)

--- 3. Attendee Performance (Your Key Metric) ---
 (Based on 45 schema-compliant samples)
Average Precision:      100.00%
Average Recall:         100.00%
Average F1-Score:       100.00%

--- 4. Timestamp Ambiguity Analysis (Confusion Matrix) ---
          PREDICTED:
          Time    |  Null (Ambiguous)
         -------------------------------
ACTUAL | Time   |  0      |  0       (Correctly Found | Missed Time)
ACTUAL | Null   |  0      |  45      (Hallucinated Ti

/opt/conda/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [16]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import icalendar
from datetime import datetime
import dateutil.parser
import sys
import os

# --- 1. Helper Function: Load Model ---
def load_trained_model(base_model_id, adapter_path):
    """Loads the base model and applies the LoRA adapters."""

    if not os.path.exists(adapter_path):
        print(f"Error: Adapter path not found: {adapter_path}", file=sys.stderr)
        print("Please ensure the model was trained and saved to this location.", file=sys.stderr)
        return None, None

    print(f"Loading base model: {base_model_id}...")
    try:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

        tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb,
            device_map="auto",
        )

        print(f"Applying LoRA adapters from: {adapter_path}...")
        model = PeftModel.from_pretrained(base_model, adapter_path)
        model = model.eval()
        print("✅ Model loaded and ready.")
        return model, tokenizer
    except Exception as e:
        print(f"Error loading model: {e}", file=sys.stderr)
        import traceback
        traceback.print_exc()
        return None, None

# --- 2. Helper Function: Prompt & Inference ---
def format_inference_prompt(email_text):
    """Formats the raw email text into the Mistral prompt template."""
    instruction = (
        "You are a precise calendar event extraction assistant. "
        "Extract the calendar event. Return ONLY a JSON object with keys: "
        "title, start_utc, end_utc, location, attendees, notes.\n"
        "- All timestamps must be ISO8601 with timezone or Z.\n"
        "- If time is ambiguous, set start_utc and end_utc to null and explain in notes.\n"
        "- Attendees should include emails from From, To, and Cc."
    )
    prompt = f"[INST] {instruction.strip()}\n\nHere is the email:\n\n{email_text.strip()} [/INST]"
    return prompt

def run_inference(model, tokenizer, email_text, max_new_tokens=1024):
    """Runs inference and attempts to parse the JSON output."""
    prompt = format_inference_prompt(email_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )

    response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    try:
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        if json_start != -1 and json_end != -1:
            json_str = response_text[json_start:json_end]
            return json.loads(json_str), response_text
        else:
            return None, response_text
    except json.JSONDecodeError:
        print(f"\nWarning: Failed to parse JSON from output:\n{response_text}\n", file=sys.stderr)
        return None, response_text

# --- 3. Helper Function: ICS Conversion ---
def convert_json_to_ics(event_json):
    """Converts the model's JSON output to an .ics file string."""
    cal = icalendar.Calendar()
    cal.add('prodid', '-//Calendar Extraction Model//')
    cal.add('version', '2.0')
    event = icalendar.Event()

    event.add('summary', event_json.get('title', 'No Title'))
    event.add('description', event_json.get('notes', ''))

    loc = event_json.get('location')
    if loc:
        event.add('location', loc)

    try:
        start_utc = event_json.get('start_utc')
        end_utc = event_json.get('end_utc')

        if start_utc and end_utc:
            dt_start = dateutil.parser.isoparse(start_utc)
            dt_end = dateutil.parser.isoparse(end_utc)
            event.add('dtstart', dt_start)
            event.add('dtend', dt_end)
        elif start_utc: # Handle if only start_utc is provided
            dt_start = dateutil.parser.isoparse(start_utc)
            event.add('dtstart', dt_start)
            event.add('dtend', dt_start + datetime.timedelta(hours=1)) # Default 1 hour
        else:
            event.add('dtstart', datetime.now().date())

    except Exception as e:
        print(f"Error parsing datetime: {e}. Skipping times.", file=sys.stderr)
        event.add('dtstart', datetime.now().date()) # Fallback

    attendees = event_json.get('attendees', [])
    for email in attendees:
        attendee = icalendar.vCalAddress(f'MAILTO:{email}')
        event.add('attendee', attendee, encode=0)

    cal.add_component(event)
    return cal.to_ical().decode('utf-8')

# --- 4. Main Execution ---
def main_test():
    BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
    ADAPTER_PATH = "./calendar_extraction_model_final"

    # Load the model
    model, tokenizer = load_trained_model(BASE_MODEL, ADAPTER_PATH)

    if model is None or tokenizer is None:
        print("Exiting due to model loading failure.")
        return

    # ** New dummy email with start and end time **
    # Note: "tomorrow" will be interpreted relative to your system's current date.
    dummy_email_text = (
        "Subject: Q4 Planning Session\n"
        "From: leelaprasad.dammalapati@sjsu.edu\n"
        "To: aishanee.sinha@sjsu.edu,debika.choudhury@sjsu.edu\n"
        "Cc: \n"
        "Bcc: \n\n"
        "Body:\n"
        "Hi all,\n\n"
        "Following up on our last discussion, let's discuss the Q4 planning session. lets meet on 6th novemner 2025\n\n"
        "Lets discuss Q4.\n\n"
        "Please confirm.\n\n"
        "Best,\n"
        "Leela"
    ).strip()

    print("="*80)
    print(f"--- INPUT EMAIL ---\n{dummy_email_text}\n{'-'*20}")

    # Run inference
    predicted_json, raw_output = run_inference(model, tokenizer, dummy_email_text)

    if predicted_json:
        print(f"--- PARSED JSON ---\n{json.dumps(predicted_json, indent=2)}\n{'-'*20}")

        # Convert to ICS
        ics_string = convert_json_to_ics(predicted_json)

        print(f"--- GENERATED .ICS FILE ---\n{ics_string}\n{'-'*20}")

        # Save to file
        output_filename = "dummy_meeting_with_end_time.ics"
        with open(output_filename, "w") as f:
            f.write(ics_string)
        print(f"✅ Successfully saved to {output_filename}")
        print("You can now import this file into your calendar.")

    else:
        print("❌ Model did not produce valid JSON.")
        print(f"--- RAW MODEL OUTPUT ---\n{raw_output}")

# --- This makes the script runnable ---
if __name__ == "__main__":
    main_test()

Loading base model: mistralai/Mistral-7B-Instruct-v0.2...


Loading checkpoint shards: 100%|██████████| 3/3 [00:29<00:00,  9.95s/it]


Applying LoRA adapters from: ./calendar_extraction_model_final...
✅ Model loaded and ready.
--- INPUT EMAIL ---
Subject: Q4 Planning Session
From: leelaprasad.dammalapati@sjsu.edu
To: aishanee.sinha@sjsu.edu,debika.choudhury@sjsu.edu
Cc: 
Bcc: 

Body:
Hi all,

Following up on our last discussion, let's discuss the Q4 planning session. lets meet on 6th novemner 2025

Lets discuss Q4.

Please confirm.

Best,
Leela
--------------------
--- PARSED JSON ---
{
  "title": "Q4 Planning Session",
  "start_utc": null,
  "end_utc": null,
  "location": null,
  "attendees": [
    "aishanee.sinha@sjsu.edu",
    "debika.choudhury@sjsu.edu",
    "leelaprasad.dammalapati@sjsu.edu"
  ],
  "notes": "Ambiguous or missing date/time. Hi all,\n\nFollowing up on our last discussion, let's discuss the Q4 planning session. lets meet on 6th novemner 2025\n\nLets discuss Q4.\n\nPlease confirm.\n\nBest,\nLeela"
}
--------------------
--- GENERATED .ICS FILE ---
BEGIN:VCALENDAR
VERSION:2.0
PRODID:-//Calendar Extrac

In [10]:
!pip install google-auth-oauthlib google-auth-httplib2 google-api-python-client icalendar python-dateutil


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 14.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [google-api-python-client]


In [11]:
# -*- coding: utf-8 -*-
"""
Import an .ics file into Google Calendar.

Usage:
  python import_ics_to_gcal.py --ics event.ics [--calendar-id primary] [--tz America/Los_Angeles] [--dry-run]

Requirements:
  pip install google-auth-oauthlib google-auth-httplib2 google-api-python-client icalendar python-dateutil

Place next to this script:
  - credentials.json  (OAuth 2.0 "Desktop app" client)
First run will create:
  - token.json
"""

import argparse
import os
import sys
from datetime import datetime, timedelta, date
from typing import List, Optional, Tuple

from dateutil import tz
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from icalendar import Calendar as ICSCalendar

SCOPES = [
    "https://www.googleapis.com/auth/calendar.events",
]

def get_credentials(credentials_file: str, token_file: str) -> Credentials:
    creds = None
    if os.path.exists(token_file):
        creds = Credentials.from_authorized_user_file(token_file, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists(credentials_file):
                raise FileNotFoundError(
                    f"Missing {credentials_file}. Download OAuth client JSON and place it next to this script."
                )
            flow = InstalledAppFlow.from_client_secrets_file(credentials_file, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_file, "w") as f:
            f.write(creds.to_json())
    return creds

def build_calendar_service(creds: Credentials):
    return build("calendar", "v3", credentials=creds)

def _dt_is_all_day(value) -> bool:
    # All-day events in ICS usually store DTSTART/DTEND as date (not datetime)
    return isinstance(value, date) and not isinstance(value, datetime)

def _ensure_tz(dt: datetime, fallback_tz: str) -> datetime:
    if dt.tzinfo is None:
        return dt.replace(tzinfo=tz.gettz(fallback_tz))
    return dt

def _datestr(d: date) -> str:
    return d.strftime("%Y-%m-%d")

def _ical_rrule_to_string(vrecur) -> Optional[str]:
    # icalendar returns a vRecur object; to_ical() yields bytes like b'FREQ=WEEKLY;BYDAY=MO'
    try:
        s = vrecur.to_ical().decode()
        return f"RRULE:{s}"
    except Exception:
        return None

def _ical_dtlist_to_lines(name: str, v) -> List[str]:
    """
    Convert EXDATE/RDATE values to Calendar API recurrence lines.
    Handles single or list; returns e.g. ["EXDATE:20251106T153000Z,20251113T153000Z"].
    """
    if v is None:
        return []
    vals = []
    try:
        # v may be a list of vDDDLists; normalize to datetimes and format in UTC (or naive)
        if not isinstance(v, list):
            v = [v]
        for itm in v:
            # itm may be a vDDDLists; iterate .dts
            dts = getattr(itm, "dts", [itm])
            for dte in dts:
                dt = dte.dt
                if isinstance(dt, date) and not isinstance(dt, datetime):
                    # All-day EXDATE/RDATE uses DATE (YYYYMMDD)
                    vals.append(dt.strftime("%Y%m%d"))
                else:
                    # Timed -> UTC stamp YYYYMMDDTHHMMSSZ
                    if dt.tzinfo is None:
                        # treat as UTC if tz is missing
                        vals.append(dt.strftime("%Y%m%dT%H%M%SZ"))
                    else:
                        dt_utc = dt.astimezone(tz.UTC)
                        vals.append(dt_utc.strftime("%Y%m%dT%H%M%SZ"))
        if vals:
            return [f"{name}:{','.join(vals)}"]
    except Exception:
        pass
    return []

def build_event_resource_from_vevent(vevent, default_tz: str) -> Tuple[dict, Optional[str]]:
    """
    Return (event_resource, iCalUID). event_resource matches Google Calendar events.insert schema.
    Supports all-day and timed events, recurrence (RRULE/EXDATE/RDATE), attendees, location, description.
    """
    summary = str(vevent.get("SUMMARY")) if vevent.get("SUMMARY") else "Imported event"
    description = str(vevent.get("DESCRIPTION")) if vevent.get("DESCRIPTION") else ""
    location = str(vevent.get("LOCATION")) if vevent.get("LOCATION") else None
    uid = str(vevent.get("UID")) if vevent.get("UID") else None

    dtstart = vevent.get("DTSTART")
    if not dtstart:
        raise ValueError("VEVENT missing DTSTART")

    dtend = vevent.get("DTEND")
    duration = vevent.get("DURATION")

    start_val = dtstart.dt
    # Normalize end
    if dtend:
        end_val = dtend.dt
    elif duration:
        # If duration is given, icalendar handles .dt for duration differently; compute if needed
        # Here we assume start is datetime/date and let icalendar calc; to be safe:
        dur = duration.dt if hasattr(duration, "dt") else duration
        end_val = start_val + dur
    else:
        # Fallback: 1 hour (timed) or +1 day (all-day)
        end_val = (start_val + timedelta(hours=1)) if isinstance(start_val, datetime) else (start_val + timedelta(days=1))

    event = {
        "summary": summary,
        "description": description[:8000],
    }
    # Determine all-day vs timed
    if _dt_is_all_day(start_val) and _dt_is_all_day(end_val):
        # For all-day, Calendar API needs exclusive end date (YYYY-MM-DD)
        event["start"] = {"date": _datestr(start_val)}
        event["end"] = {"date": _datestr(end_val)}
    else:
        # Ensure tz
        start_dt = _ensure_tz(start_val if isinstance(start_val, datetime) else datetime.combine(start_val, datetime.min.time()), default_tz)
        end_dt = _ensure_tz(end_val if isinstance(end_val, datetime) else datetime.combine(end_val, datetime.min.time()), default_tz)
        event["start"] = {"dateTime": start_dt.isoformat(), "timeZone": default_tz}
        event["end"] = {"dateTime": end_dt.isoformat(), "timeZone": default_tz}

    if location:
        event["location"] = location

    # Attendees
    attendees = []
    for att in vevent.getall("ATTENDEE") or []:
        s = str(att)
        s = s.strip()
        # Common forms: 'MAILTO:user@example.com' or 'mailto:user@example.com'
        if "MAILTO:" in s.upper():
            email = s.split(":", 1)[1]
            attendees.append({"email": email})
    if attendees:
        # Deduplicate by email
        seen = {}
        for a in attendees:
            seen[a["email"].lower()] = a
        event["attendees"] = list(seen.values())

    # Recurrence: RRULE (+ optional EXDATE/RDATE)
    recur_lines: List[str] = []
    rrule = vevent.get("RRULE")
    if rrule:
        rrule_line = _ical_rrule_to_string(rrule)
        if rrule_line:
            recur_lines.append(rrule_line)
    exdate = vevent.get("EXDATE")
    rdate = vevent.get("RDATE")
    recur_lines.extend(_ical_dtlist_to_lines("EXDATE", exdate))
    recur_lines.extend(_ical_dtlist_to_lines("RDATE", rdate))
    if recur_lines:
        event["recurrence"] = recur_lines

    return event, uid

def find_existing_by_uid(cal_svc, calendar_id: str, uid: str) -> Optional[dict]:
    try:
        resp = cal_svc.events().list(calendarId=calendar_id, iCalUID=uid, maxResults=1, singleEvents=True).execute()
        items = resp.get("items", [])
        return items[0] if items else None
    except HttpError:
        return None

def insert_event(cal_svc, calendar_id: str, event: dict, send_updates: str = "all") -> dict:
    return cal_svc.events().insert(calendarId=calendar_id, body=event, sendUpdates=send_updates).execute()

def import_ics(ics_path: str, calendar_id: str, default_tz: str, dry_run: bool) -> None:
    with open(ics_path, "rb") as f:
        cal = ICSCalendar.from_ical(f.read())

    vevents = [c for c in cal.walk() if getattr(c, "name", None) == "VEVENT"]
    if not vevents:
        print("No VEVENTs found in the .ics file.")
        return

    print(f"Found {len(vevents)} event(s) in {ics_path}.")

    creds = get_credentials("credentials.json", "token.json")
    cal_svc = build_calendar_service(creds)

    created = 0
    skipped = 0
    for idx, v in enumerate(vevents, 1):
        try:
            event_resource, uid = build_event_resource_from_vevent(v, default_tz)
        except Exception as e:
            print(f"[{idx}] Skipping VEVENT (parse error): {e}")
            continue

        # De-duplication by iCalUID if available
        existing = None
        if uid:
            existing = find_existing_by_uid(cal_svc, calendar_id, uid)

        if existing:
            print(f"[{idx}] Skipped (already exists) — Summary: {existing.get('summary')} | Link: {existing.get('htmlLink')}")
            skipped += 1
            continue

        if dry_run:
            print(f"[{idx}] DRY RUN — would create event:")
            print(f"      Summary: {event_resource.get('summary')}")
            print(f"      Start:   {event_resource.get('start')}")
            print(f"      End:     {event_resource.get('end')}")
            if "recurrence" in event_resource:
                print(f"      Recurs:  {event_resource['recurrence']}")
            if "attendees" in event_resource:
                print(f"      Attendees: {[a['email'] for a in event_resource['attendees']]}")
            continue

        try:
            evt = insert_event(cal_svc, calendar_id, event_resource, send_updates="all")
            print(f"[{idx}] Created — {evt.get('summary')} | Link: {evt.get('htmlLink')}")
            created += 1
        except HttpError as e:
            print(f"[{idx}] Calendar API error: {e}")

    print(f"\nDone. Created: {created}, Skipped (existing): {skipped}, Total parsed: {len(vevents)}")

def main():
    parser = argparse.ArgumentParser(description="Create Google Calendar meeting(s) from an .ics file.")
    parser.add_argument("--ics", required=True, help="Path to the .ics file")
    parser.add_argument("--calendar-id", default="primary", help="Target calendar ID (default: primary)")
    parser.add_argument("--tz", default="America/Los_Angeles", help="Default timezone if DTSTART/DTEND lack TZID")
    parser.add_argument("--dry-run", action="store_true", help="Preview events without creating them")
    args = parser.parse_args()

    if not os.path.exists(args.ics):
        print(f".ics not found: {args.ics}", file=sys.stderr)
        sys.exit(1)

    import_ics(args.ics, args.calendar_id, args.tz, args.dry_run)

if __name__ == "__main__":
    main()


/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.13) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
usage: ipykernel_launcher.py [-h] --ics ICS [--calendar-id CALENDAR_ID]
                             [--tz TZ] [--dry-run]
ipykernel_launcher.py: error: the following arguments are required: --ics


SystemExit: 2

/opt/conda/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
